In [1]:
import os
import json
import pandas as pd

from pathlib import Path
from roxy.helpers import RoxyHelpers
from datetime import datetime, timezone
from maomao.physicochemical_characteristics.description_sequence import *

# Compatibility bridge for Roxy 0.1.0 with pandas >= 3.0.
if not hasattr(pd.DataFrame, "applymap"):
    pd.DataFrame.applymap = pd.DataFrame.map

/home/nicole/miniconda3/envs/maomao_resource/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Sequence descriptor report

This notebook computes physicochemical and sequence-derived descriptors for every peptide included in the MAOMAO sequence-level resource.

Unlike an endpoint-specific characterization, this analysis does not define a single target variable. The toxicity endpoints are preserved as independent evidence columns associated with each sequence:

- `toxic`
- `cytotoxic`
- `hemolytic`
- `cytolysis`
- `neurotoxic`
- `embryotoxic`
- `ichthyotoxic`

Global sequence descriptors are calculated using Roxy and combined with the original MAOMAO sequence identifier, amino-acid sequence, and toxicity evidence columns.

Dimensionality-reduction methods and graphical analyses are disabled because the purpose of this notebook is exclusively to generate a reusable sequence descriptor table.

The notebook produces:

- `sequence_descriptors.csv`: sequence identifiers, sequences, toxicity evidence columns, and computed descriptors;
- `metadata.json`: processing settings, dataset dimensions, descriptor names, and endpoint evidence distributions.

- Definition of general variables

In [2]:
path_data = "../../processed_data/processed_data"
path_export = "../../dataset_characterization"

input_file = f"{path_data}/maomao_sequence_pivot.csv"
output_file = f"{path_export}/sequence_descriptors.csv"
metadata_file = f"{path_export}/metadata.json"

#### Description of the toxic effects dataset by label

- Read data

In [3]:
df_data = pd.read_csv(input_file)
df_data

,id,sequence,toxic,cytotoxic,hemolytic,cytolysis,neurotoxic,embryotoxic,ichthyotoxic
0,sha256_c70d828a62440a19acb4c85c0d1cff29a5b340a...,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,2,0,2,999,999,999,999
1,sha256_cb0e712239fad2596dfffa24ab580259aba066d...,AAAAAAAAAGETS,999,0,0,999,999,999,999
2,sha256_966434f9646239a8cd0a9375d1036fb4328d803...,AAAAAAAAAK,999,999,0,999,999,999,999
3,sha256_1de8b390ef5a86661bf0da4babf5a6059743fb6...,AAAAAAAIKMLMDLVNERIMALNKKAKK,0,0,0,999,999,999,999
4,sha256_42c3b829be38387546e22839e4d70387e2af55c...,AAAAARRRIRKQAHAHSK,0,0,0,999,999,999,999
...,...,...,...,...,...,...,...,...,...
71696,sha256_1b72221eb2481c00d2b3035378ac777f29b8e35...,YYVDLQNR,2,999,999,999,999,999,999
71697,sha256_fc79c43344c8a599efa1c57c80225185ca0342c...,YYVWIGLRWVNIDCVEGNWSDYSSVSYENLVR,1,999,999,999,999,999,999
71698,sha256_e460b0b56fef4833c831a3083bb151aa7608912...,YYYAAGRKRKKRT,0,0,0,999,999,999,999
71699,sha256_bc4076682461bf7840e8705e1ab3a56eb1077cd...,YYYELLVDLL,1,1,1,999,999,999,999


- Check columns

In [4]:
label_cols = [
    "toxic",
    "cytotoxic",
    "hemolytic",
    "cytolysis",
    "neurotoxic",
    "embryotoxic",
    "ichthyotoxic",
]

required_cols = [
    "id",
    "sequence",
    *label_cols,
]

missing_cols = [
    column
    for column in required_cols
    if column not in df_data.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )

- Describe sequences with `RoxyHelpers.describe_sequences`

In [5]:
result = RoxyHelpers.describe_sequences(
    df=df_data,
    seq_col="sequence",
    y=None, # MAOMAO is multilabel, so no single target is used.
    dataset_name="maomao",

    # False: only global sequence descriptors.
    # True: global descriptors + AAIndex descriptors, when available.
    use_aaindex=False,
    # Disable PCA, UMAP, t-SNE, etc.
    project_method=None,
)

- Build final descriptor table

In [7]:
columns = (
    df_data[["id", "sequence", *label_cols,]]
    .reset_index(drop=True)
)
descriptors = result.X.reset_index(drop=True)
df_descriptors = pd.concat([columns, descriptors,], axis=1,)
df_descriptors.head()

,id,sequence,toxic,cytotoxic,hemolytic,cytolysis,neurotoxic,embryotoxic,ichthyotoxic,length,...,boman_index,net_charge_pH,fcr,ncpr,donors_per_residue,acceptors_per_residue,aa_entropy,lc_k1,lc_k2,lc_k3
0,sha256_c70d828a62440a19acb4c85c0d1cff29a5b340a...,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,2,0,2,999,999,999,999,33.0,...,-0.020303,2.998495,0.181818,0.090863,0.242424,0.151515,2.990623,0.600000,0.687500,0.774194
1,sha256_cb0e712239fad2596dfffa24ab580259aba066d...,AAAAAAAAAGETS,999,0,0,999,999,999,999,13.0,...,-0.057692,-1.000241,0.076923,-0.076942,0.153846,0.230769,1.505876,0.384615,0.416667,0.454545
2,sha256_966434f9646239a8cd0a9375d1036fb4328d803...,AAAAAAAAAK,999,999,0,999,999,999,999,10.0,...,0.054000,0.997668,0.100000,0.099767,0.100000,0.000000,0.468996,0.200000,0.222222,0.250000
3,sha256_1de8b390ef5a86661bf0da4babf5a6059743fb6...,AAAAAAAIKMLMDLVNERIMALNKKAKK,0,0,0,999,999,999,999,28.0,...,-0.195000,3.998900,0.285714,0.142818,0.285714,0.142857,2.891328,0.500000,0.777778,0.846154
4,sha256_42c3b829be38387546e22839e4d70387e2af55c...,AAAAARRRIRKQAHAHSK,0,0,0,999,999,999,999,18.0,...,-0.397222,5.997339,0.444444,0.333186,0.555556,0.222222,2.411509,0.388889,0.705882,0.875000


- Create metadata

In [8]:
label_distributions = {}

for label_col in label_cols:
    counts = (
        df_data[label_col]
        .value_counts(dropna=False)
        .sort_index()
    )

    label_distributions[label_col] = {
        str(value): int(count)
        for value, count in counts.items()
    }

In [9]:
# Descriptor metadata
metadata = {
    "dataset_name": "maomao",
    "description": (
        "Physicochemical and sequence-derived descriptor table "
        "generated from the MAOMAO sequence-level toxicity resource."
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "input": {
        "file": str(input_file),
        "id_column": "id",
        "sequence_column": "sequence",
        "label_columns": label_cols,
        "n_rows": int(df_data.shape[0]),
        "n_columns": int(df_data.shape[1]),
    },

    "descriptor_generation": {
        "software": "Roxy",
        "helper": "RoxyHelpers.describe_sequences",
        "feature_keys": list(result.feature_keys),
        "use_aaindex": False,
        "project_method": None,
        "target_column": None,
        "target_handling": (
            "No single target was used because MAOMAO contains "
            "multiple toxicity evidence columns."
        ),
    },

    "label_encoding": {
        "0": "negative",
        "1": "positive",
        "2": "ambiguous",
        "3": "unlabeled",
        "999": "no_information",
    },

    "output": {
        "descriptor_table": str(output_file),
        "metadata_file": str(metadata_file),
        "n_rows": int(df_descriptors.shape[0]),
        "n_columns": int(df_descriptors.shape[1]),
        "n_descriptor_columns": int(
            descriptors.shape[1]
        ),
        "descriptor_columns": (
            descriptors.columns
            .astype(str)
            .tolist()
        ),
    },

    "data_quality": {
        "missing_ids": int(
            df_data["id"].isna().sum()
        ),
        "missing_sequences": int(
            df_data["sequence"].isna().sum()
        ),
        "duplicated_ids": int(
            df_data["id"].duplicated().sum()
        ),
        "duplicated_sequences": int(
            df_data["sequence"]
            .duplicated()
            .sum()
        ),
    },

    "label_distributions": label_distributions,
}

- Save descriptors

In [10]:
Path(path_export).mkdir(
    parents=True,
    exist_ok=True,
)

df_descriptors.to_csv(
    output_file,
    index=False,
)

In [11]:
# Save metadata
Path(metadata_file).write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

2935

In [12]:
print(f"Descriptor table saved to: {output_file}")
print(f"Shape: {df_descriptors.shape}")
print(f"Descriptor columns: {descriptors.shape[1]}")

Descriptor table saved to: ../../dataset_characterization/sequence_descriptors.csv
Shape: (71701, 50)
Descriptor columns: 41
